# Lekcja 09 - Wzorzec projektowy metapoznania


## Setup

This notebook demonstrates the Metacognition design pattern using the Microsoft Agent Framework.

**Prerequisites:**
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Czym jest metapoznanie?

Metapoznanie to **myślenie o myśleniu**. W kontekście agentów sztucznej inteligencji oznacza to tworzenie agentów, którzy potrafią:

- **Samorefleksję** nad własnymi wynikami i procesem rozumowania
- **Wykrywać błędy** i radzić sobie z nimi w sposób łagodny, zamiast zawieszać się bez słowa
- **Oceniać**, czy ich odpowiedzi są pełne i pomocne
- **Dostosowywać** swoją strategię, gdy początkowe podejście zawodzi (np. przejść do systemu zapasowego)

Agent metapoznawczy nie tylko odpowiada na pytania — monitoruje własną efektywność i dostosowuje się na bieżąco.


## Narzędzia podstawowe i zapasowe

Powszechnym wzorcem metapoznawczym jest **strategia zapasowa**. Agent najpierw próbuje użyć narzędzia podstawowego; jeśli to się nie powiedzie (np. błąd 404), agent rozpoznaje porażkę i transparentnie przełącza się na narzędzie zapasowe.

Odzwierciedla to systemy z rzeczywistego świata, gdzie usługi podstawowe mogą być niedostępne, a agenci muszą samodzielnie zdiagnozować problem, zanim wybiorą alternatywną ścieżkę.

Poniżej definiujemy dwa narzędzia do wyszukiwania lotów:
- **Podstawowe** — obejmuje Paryż, Tokio i Barcelonę
- **Zapasowe** — obejmuje Berlin, Sydney i Nowy Jork


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agent Samooceniający się z Odzyskiwaniem po Błędach

Poniższy agent ma za zadanie najpierw spróbować użyć głównego systemu lotu, rozpoznać awarie i przejrzyście przełączyć się na system zapasowy. Po każdej odpowiedzi krótko ocenia, czy w pełni odpowiedział na pytanie użytkownika.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Wzorzec samooceny

Innym aspektem metapoznania jest **samoocena**: niezależny agent (lub ten sam agent w drugiej turze) przegląda odpowiedź pod kątem kompletności, dokładności i przydatności.

Poniżej tworzymy agenta `ResponseEvaluator`, który ocenia odpowiedzi agenta podróży według trzech wymiarów.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Podsumowanie

W tej lekcji nauczyłeś się, jak tworzyć **agentów metapoznawczych** za pomocą Microsoft Agent Framework:

- **Samorefleksja**: Agenci, którzy monitorują własne rozumowanie i transparentnie komunikują przebieg wydarzeń.
- **Odporność na błędy dzięki mechanizmom zapasowym**: Wzorzec narzędzia głównego + zapasowego, gdzie agent wykrywa awarie (np. błędy 404) i automatycznie próbuje alternatywnego źródła.
- **Samoocena**: Oddzielny agent oceniający, który przyznaje odpowiedziom oceny za kompletność, dokładność i przydatność.

Te wzorce sprawiają, że agenci są bardziej odporni, transparentni i godni zaufania — cechy kluczowe dla wdrożeń produkcyjnych.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
